In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import torch


dataset = load_dataset(
    "dim/hendrycks_math_train_1k_DeepSeek-R1-Distill-Qwen-1.5B_max_len_4096_greedy"
)
dataset = dataset["train"].train_test_split(
    # test_size=250,
    test_size=350,
    # test_size=999,
    # test_size=1,
    seed=42,
)
dataset = dataset["test"].filter(lambda x: x["model_answer"].count("</think>") == 1)

model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},
    # attn_implementation="sdpa",
    attn_implementation="flash_attention_2",
)
model.requires_grad_(False)
tokenizer = AutoTokenizer.from_pretrained(model_name)

### Кодируем части текста в вектора

In [2]:
!pip install more_itertools -q

In [ ]:
[1, 2, 3, 4, 5, 6][-2:]

[5, 6]

In [168]:
from more_itertools import chunked
from tqdm import tqdm
import itertools

start_item = 100
for i in range(start_item, len(dataset)):
    print(f"{i}/{len(dataset)}")
    model_answer = dataset[i]["model_answer"]
    tokens = tokenizer.encode(
        dataset[i]["model_answer"],
        add_special_tokens=False,
    )
    mem_tokens = 8
    encode_size = mem_tokens * 4
    tokens_chunks = list(chunked(tokens, encode_size))
    for chunk_part in range(2, len(tokens_chunks)):
        train_part = tokens_chunks[:chunk_part]
        train_part = list(itertools.chain(*train_part))
        train_part_full_str = tokenizer.decode(
            train_part,
            add_special_tokens=False,
        )
        train_part_target_str = tokenizer.decode(
            train_part[-encode_size:][1:],
            add_special_tokens=False,
        )
        train_part = torch.tensor(
            train_part,
            device="cuda",
        ).unsqueeze(0)
        labels_part = train_part.clone()
        labels_part[:, :-encode_size] = -100
        # labels_part[]
        # print(train_part)
        # print(labels_part)
        compression_tensor_param = torch.nn.Parameter(
            torch.rand(
                mem_tokens,
                model.get_input_embeddings().weight.shape[1],
                device="cuda",
            ).unsqueeze(0),
            requires_grad=True,
        )
        optimizer = torch.optim.AdamW(
            [
                compression_tensor_param,
            ],
            lr=0.1,
        )
        compression_tensor = compression_tensor_param.repeat(
            1, encode_size // mem_tokens, 1
        )
        prev_tokens = train_part[:, :-encode_size].clone()
        prev_embeds = model.get_input_embeddings()(prev_tokens)
        input_embeds = torch.cat(
            [
                prev_embeds,
                compression_tensor,
            ],
            dim=1,
        )
        epoch_amount = 50
        dtype = torch.bfloat16
        for epoch in tqdm(range(epoch_amount)):
            compression_tensor = compression_tensor_param.repeat(
                1, encode_size // mem_tokens, 1
            )
            # print(compression_tensor_param)
            input_embeds = torch.cat(
                [
                    prev_embeds,
                    compression_tensor,
                ],
                dim=1,
            ).to(dtype)
            model_predicts = model(
                inputs_embeds=input_embeds,
                labels=labels_part,
            )

            compression_loss = model_predicts.loss
            compression_loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            # print(
            #     "compression_loss",
            #     compression_loss,
            #     epoch,
            #     # len(tokenizer.encode(model_answer)),
            # )
        prediction_str = tokenizer.decode(
            model_predicts.logits.argmax(-1)[:, -encode_size:][-1]
        )
        # print("train_part_full_str", train_part_full_str)
        # print("train_part_target_str", train_part_target_str)
        # print("prediction_str", prediction_str)
        correct_reconstruction = (
            tokenizer.decode(
                model_predicts.logits.argmax(-1)[:, -encode_size:][-1][:-1]
            )
            == train_part_target_str
        )
        print("correct_reconstruction", correct_reconstruction)
        # break

    break

100/209


  0%|          | 0/50 [00:00<?, ?it/s]

100%|██████████| 50/50 [00:01<00:00, 37.80it/s]


correct_reconstruction True


100%|██████████| 50/50 [00:01<00:00, 37.02it/s]


...

correct_reconstruction True


100%|██████████| 50/50 [00:04<00:00, 11.90it/s]


correct_reconstruction True


 46%|████▌     | 23/50 [00:02<00:02, 11.46it/s]


KeyboardInterrupt: 

In [150]:
tokenizer.decode(
    train_part[:, -encode_size:][-1][1:],
    # train_part[:, -encode_size:][-1][:],
    add_special_tokens=False,
)

' i, O = -4, P = -i, and S = 2 + 4i. Hmm, let me write that down to make'

In [138]:
train_part[:, -encode_size:][-1]

tensor([  488,   600,    11,   506,   284,   481,    19,    11,   393,   284,
          481,    72,    11,   323,   328,   284,   220,    17,   488,   220,
           19,    72,    13, 88190,    11,  1077,   752,  3270,   429,  1495,
          311,  1281], device='cuda:0')

In [139]:
labels_part[labels_part != -100]

tensor([   17,   488,   600,    11,   506,   284,   481,    19,    11,   393,
          284,   481,    72,    11,   323,   328,   284,   220,    17,   488,
          220,    19,    72,    13, 88190,    11,  1077,   752,  3270,   429,
         1495,   311,  1281], device='cuda:0')

In [64]:
tokenizer.decode(train_part[:, :][-1])

'Okay, so I have this problem here where I need to find the value of A minus O plus P plus S. The given values are A = 2 + i, O = -4, P = -i, and S = 2 + 4i. Hmm, let me write that down to make'

In [78]:
tokenizer.decode(train_part[:, :-encode_size][-1])

'Okay, so I have this problem here where I need to find the value of'

In [115]:
model_predicts.logits.argmax(-1)[:, -encode_size:][-1]

tensor([  600,    11,   506,   284,   481,    19,    11,   393,   284,   481,
           72,    11,   323,   328,   284,   220,    17,   488,   220,    19,
           72,    13, 88190,    11,  1077,   752,  3270,   429,  1495,   311,
         1281,   220], device='cuda:0')

In [145]:
tokenizer.decode(model_predicts.logits.argmax(-1)[:, -encode_size:][-1][:-1])

' i, O = -4, P = -i, and S = 2 + 4i. Hmm, let me write that down to make'

In [146]:
train_part_target_str

' i, O = -4, P = -i, and S = 2 + 4i. Hmm, let me write that down to make'

In [147]:
tokenizer.decode(
    model_predicts.logits.argmax(-1)[:, -encode_size:][-1][:-1]
) == train_part_target_str

True

In [ ]:
tokenizer.decode(model_predicts.logits.argmax(-1)[:, -encode_size:][-1])

' i, O = -4, P = -i, and S = 2 + 4i. Hmm, let me write that down to make t'

In [ ]:
tokenizer.decode(train_part[:, :-encode_size][-1])

'Okay, so I have this problem here where I need to find the value of A minus O plus P plus S. The given values are A = 2'

In [122]:
train_part_full_str

'Okay, so I have this problem here where I need to find the value of A minus O plus P plus S. The given values are A = 2 + i, O = -4, P = -i, and S = 2 + 4i. Hmm, let me write that down to make'

In [13]:
model_predicts.logits.argmax(-1)[-1:, :-encode_size][-1]

tensor([22573,   773,   358,   614,   419,  3491,  1588,    25,   358,  1184,
          311,  1477,   279,   897,   315,   856,    13,   425,    11,   393,
        27283,  1207, 27283,   576,  3491,  2750,   525,   362,   374,   220],
       device='cuda:0')